# Incremental Processing: New Files, Checkpoints, Watermarks, Upserts, Late Data

## What you'll practice

Full reloads don't scale. Incremental pipelines process only what's new — and must stay correct when files are re-delivered, data arrives late, or a run crashes midway.
`datasets/daily/` simulates a week of partitions to practise on.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/etl/test_08_05_incremental_processing.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/08_data_engineering_python/05_incremental_processing.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Which files are new?

**Implement:** `files_to_process(available: list[str], processed: set[str]) -> list[str]`

Return available file keys not yet processed, sorted naturally by partition then name (plain sorting works for `dt=YYYY-MM-DD/...` keys), without duplicates.

**Input:** `available: list[str], processed: set[str]`

**Expected output:** `list[str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
files_to_process([], set())
# -> []
```

**Check:** `pytest tests/etl/test_08_05_incremental_processing.py -k test_files_to_process`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · A durable JSON checkpoint

**Implement:** `class Checkpoint`

`Checkpoint(path)` loads existing state (or starts empty): `processed: dict[file_key, checksum]` and `watermark: str | None`. Methods: `is_processed(key, checksum=None)`
(if a checksum is given, it must match too), `mark(key, checksum)`, `advance(watermark)` (only moves forward), `save()` — atomic (temp file + `os.replace`). Saved JSON:
`{"processed": {...}, "watermark": ..., "updated_at": ISO UTC}`. A corrupt checkpoint file → `ValueError` (never silently reset: that would reprocess everything!).

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_checkpoint_roundtrip, test_checkpoint_corrupt` in `tests/etl/test_08_05_incremental_processing.py`.

**Check:** `pytest tests/etl/test_08_05_incremental_processing.py -k "test_checkpoint_roundtrip or test_checkpoint_corrupt"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Watermark filtering

**Implement:** `select_new(records: list[dict], watermark: str | None, ts_field: str, lookback_seconds: int = 0) -> tuple[list[dict], str | None]`

Return records with `ts > watermark - lookback` (ISO UTC strings with `Z`; `None` watermark → everything) and the **new watermark** = max `ts` among *all* inputs (not just selected),
never lower than the old watermark. A lookback re-reads a safety window to catch slightly-late records (downstream must then be idempotent).

**Input:** `records: list[dict], watermark: str | None, ts_field: str, lookback_seconds: int = 0`

**Expected output:** `tuple[list[dict], str | None]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
select_new([], None, 'ts')
# -> ([], None)
```

**Check:** `pytest tests/etl/test_08_05_incremental_processing.py -k test_select_new`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Upserts with stale-update protection

**Implement:** `upsert(target: dict, changes: list[dict], key: str, version_field: str) -> dict`

`target` maps key → current row. Apply `changes` **in place**: insert unknown keys; update when the change's `version_field` is **greater** than the stored row's; otherwise
skip it as stale (equal version = duplicate delivery, also skipped). A change with `"_op": "delete"` removes the key if its version is newer. Return counts
`{"inserted", "updated", "deleted", "skipped_stale"}`. Order of `changes` must not matter for the final state (out-of-order CDC events).

**Input:** `target: dict, changes: list[dict], key: str, version_field: str`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_upsert, test_upsert_order_independent_with_deletes` in `tests/etl/test_08_05_incremental_processing.py`.

**Check:** `pytest tests/etl/test_08_05_incremental_processing.py -k "test_upsert or test_upsert_order_independent_with_deletes"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Late data and a reprocessing window · 🧩 Mini-project

**Implement:** `update_daily_aggregates(state: dict, events: list[dict], allowed_lateness_days: int = 2) -> tuple[dict, list[str], int]`

Maintain daily revenue aggregates incrementally. `state = {"watermark": "YYYY-MM-DD" | None, "days": {date: {"revenue": float, "events": int, "event_ids": [...]}}}`.
For each event `{event_id, date: "YYYY-MM-DD", amount}`:
- if `date < watermark - allowed_lateness_days` → **drop** it (too late; count it);
- if `event_id` already counted for that date → ignore (idempotent replays);
- otherwise add it to that day.
After the batch, advance the watermark to the max event date seen (never backwards). Return `(new_state, sorted list of changed dates, dropped_count)` without mutating the input
state. Revenue rounded to 2 decimals in the output. This is how Spark Structured Streaming's watermarks bound late data.

**Input:** `state: dict, events: list[dict], allowed_lateness_days: int = 2`

**Expected output:** `tuple[dict, list[str], int]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_update_daily_aggregates` in `tests/etl/test_08_05_incremental_processing.py`.

**Check:** `pytest tests/etl/test_08_05_incremental_processing.py -k test_update_daily_aggregates`

**YOUR CODE ↓**